# Segment a MIMIC-CXR image with CXAS

This notebook runs the repository's pretrained `UNet_ResNet50_default` model on one frontal MIMIC-CXR JPEG. It saves all 159 binary anatomy masks in one compressed `.npz` file and writes a PNG preview for the lungs and heart. The weights are downloaded automatically on the first run.

> CXAS output is intended for research use and is not a clinical diagnosis.

## 1. Locate the clone and install missing dependencies

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/home/data2/chk/workspace/2026/08/04/medical_world_model/code/ChestXRayAnatomySegmentation"),
]
REPO_ROOT = next(
    (path.resolve() for path in repo_candidates if (path / "cxas").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the ChestXRayAnatomySegmentation clone.")

# Probe in a child process so a failed import cannot leave partial modules in this kernel.
required_imports = [
    "torch", "torchvision", "PIL", "numpy", "matplotlib",
    "SimpleITK", "gdown", "cv2", "colorcet",
    "pydicom_seg", "pycocotools",
]
probe_code = "; ".join(f"import {name}" for name in required_imports)
probe_code += (
    "; import inspect, gdown"
    "; assert 'fuzzy' in inspect.signature(gdown.download).parameters"
)
probe = subprocess.run(
    [sys.executable, "-c", probe_code],
    capture_output=True,
    text=True,
)
if probe.returncode != 0:
    print("Installing CXAS and notebook dependencies into the current environment...")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install",
            "-e", str(REPO_ROOT), "matplotlib", "gdown<6",
        ]
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT}")
print(f"Python:     {sys.executable}")

## 2. Select and inspect a frontal MIMIC-CXR image

The selected file is present in the supplied MIMIC tree and its MIMIC metadata records it as an erect postero-anterior (PA) view. Replace `IMAGE_PATH` with another `.jpg`, `.png`, or `.dcm` file if needed; a frontal PA/AP image is the most appropriate input for this example.

In [ ]:
from IPython.display import display
from PIL import Image

MIMIC_ROOT = Path("/home/data1/data/MIMIC/MIMIC_CXR")
IMAGE_PATH = MIMIC_ROOT / (
    "files/p17/p17936363/s52670819/"
    "066fbf06-16306bf8-94e94f61-8a4a534a-0f449d84.jpg"
)
OUTPUT_DIR = REPO_ROOT / "outputs" / "mimic_segmentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"MIMIC image not found: {IMAGE_PATH}")

image = Image.open(IMAGE_PATH).convert("L")
print(f"Input: {IMAGE_PATH}")
print(f"Size:  {image.size[0]} x {image.size[1]} pixels")
display(image.resize((424, 512)))

## 3. Load the pretrained model and run segmentation

On the first run, the notebook downloads the checkpoint published in the [author's Hugging Face demo space](https://huggingface.co/spaces/cmseibold/cxas-demo/tree/main) to the cache location expected by CXAS (`~/.cxas/weights/` by default). Inference uses the first CUDA GPU when one is available, otherwise CPU.

In [ ]:
import os
import numpy as np
import torch

from cxas import CXAS
from cxas.label_mapper import id2label_dict

checkpoint_root = Path(os.environ.get("CXAS_PATH", Path.home()))
checkpoint_path = checkpoint_root / ".cxas/weights/UNet_ResNet50_default.pth"
checkpoint_url = (
    "https://huggingface.co/spaces/cmseibold/cxas-demo/resolve/main/"
    "weights/.cxas/weights/UNet_ResNet50_default.pth?download=true"
)
if not checkpoint_path.is_file():
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading model checkpoint to {checkpoint_path}")
    torch.hub.download_url_to_file(checkpoint_url, str(checkpoint_path), progress=True)

device_arg = "0" if torch.cuda.is_available() else "cpu"
print("Inference device:", "cuda:0" if device_arg == "0" else "cpu")

model = CXAS(model_name="UNet_ResNet50_default", gpus=device_arg)
predictions = model.process_file(filename=str(IMAGE_PATH))

masks = predictions["segmentation_preds"][0].detach().cpu().numpy().astype(bool)
class_names = np.array(
    [id2label_dict[str(index)] for index in range(masks.shape[0])],
    dtype=str,
)
print(f"Mask tensor: {masks.shape} ({masks.dtype})")
print(f"Non-empty classes: {(masks.reshape(masks.shape[0], -1).any(axis=1)).sum()} / {masks.shape[0]}")

## 4. Save the masks and visualize key anatomy

The compressed result stores the mask in `[class, height, width]` order at the model's 512 × 512 inference resolution, together with its class names and source path. This is substantially smaller than materializing all 159 masks at the original image resolution.

In [ ]:
from cxas.visualize import visualize_mask

mask_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_cxas_masks_512.npz"
np.savez_compressed(
    mask_path,
    masks=masks,
    class_names=class_names,
    source_path=np.array(str(IMAGE_PATH)),
    threshold=np.float32(0.5),
)

overlay = visualize_mask(
    class_names=["right lung", "left lung", "heart"],
    mask=masks,
    image=predictions["orig_data"],
    img_size=512,
    cat=True,
    axis=1,
)
overlay_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_cxas_overlay.png"
overlay.save(overlay_path)

print(f"Saved masks:   {mask_path}")
print(f"Saved preview: {overlay_path}")
display(overlay)

## 5. Inspect individual masks

In [ ]:
import matplotlib.pyplot as plt

classes_to_show = [
    "right lung",
    "left lung",
    "heart",
    "cardiomediastinum",
    "trachea",
    "diaphragm",
]
name_to_index = {name: index for index, name in enumerate(class_names)}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, class_name in zip(axes.flat, classes_to_show):
    axis.imshow(masks[name_to_index[class_name]], cmap="gray")
    axis.set_title(class_name)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 6. Segment ten additional MIMIC-CXR examples

This section selects ten more frontal images (PA or AP) from distinct patients, excluding the first example. Each example highlights a different anatomical area. It saves a compressed 159-class mask and focused overlay for every image, along with a CSV summary and contact sheet.

In [ ]:
import csv

metadata_path = MIMIC_ROOT / "mimic-cxr-2.0.0-metadata.csv"
excluded_dicom_ids = {IMAGE_PATH.stem}
excluded_subject_ids = {"17936363"}
additional_examples = []
selected_subject_ids = set()

with metadata_path.open(newline="") as handle:
    for row in csv.DictReader(handle):
        if row["ViewPosition"] not in {"PA", "AP"}:
            continue
        if row["dicom_id"] in excluded_dicom_ids:
            continue
        if row["subject_id"] in excluded_subject_ids or row["subject_id"] in selected_subject_ids:
            continue

        subject_id = row["subject_id"]
        image_path = (
            MIMIC_ROOT
            / "files"
            / f"p{subject_id[:2]}"
            / f"p{subject_id}"
            / f"s{row['study_id']}"
            / f"{row['dicom_id']}.jpg"
        )
        if not image_path.is_file():
            continue

        additional_examples.append(
            {
                "dicom_id": row["dicom_id"],
                "subject_id": subject_id,
                "study_id": row["study_id"],
                "view": row["ViewPosition"],
                "image_path": image_path,
            }
        )
        selected_subject_ids.add(subject_id)
        if len(additional_examples) == 10:
            break

if len(additional_examples) != 10:
    raise RuntimeError(f"Found only {len(additional_examples)} suitable frontal images.")

focus_areas = [
    ("lung fields", ["right lung", "left lung"]),
    ("heart", ["heart"]),
    ("lung zones", [
        "right upper zone lung", "right mid zone lung", "right lung base",
        "right apical zone lung", "left upper zone lung", "left mid zone lung",
        "left lung base", "left apical zone lung",
    ]),
    ("ribs", ["ribs super"]),
    ("thoracic spine", ["thoracic spine"]),
    ("shoulder girdle", [
        "clavicle left", "clavicle right", "scapula left", "scapula right",
    ]),
    ("diaphragm", ["left hemidiaphragm", "right hemidiaphragm"]),
    ("mediastinal vessels", ["aortic arch", "descending aorta", "pulmonary artery"]),
    ("airway", ["trachea", "tracheal bifurcation"]),
    ("upper abdomen", ["stomach", "liver"]),
]
for example, (focus_name, focus_classes) in zip(additional_examples, focus_areas):
    example["focus_name"] = focus_name
    example["focus_classes"] = focus_classes

for index, example in enumerate(additional_examples, start=1):
    print(
        f"{index:02d}  {example['view']:2s}  {example['focus_name']:20s}  "
        f"subject {example['subject_id']}  {example['image_path'].name}"
    )

In [ ]:
TEN_OUTPUT_DIR = OUTPUT_DIR / "ten_examples"
TEN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_rows = []
example_overlays = []

for index, example in enumerate(additional_examples, start=1):
    batch_predictions = model.process_file(filename=str(example["image_path"]))
    batch_masks = (
        batch_predictions["segmentation_preds"][0]
        .detach()
        .cpu()
        .numpy()
        .astype(bool)
    )

    batch_mask_path = TEN_OUTPUT_DIR / f"{example['dicom_id']}_cxas_masks_512.npz"
    np.savez_compressed(
        batch_mask_path,
        masks=batch_masks,
        class_names=class_names,
        source_path=np.array(str(example["image_path"])),
        threshold=np.float32(0.5),
    )

    batch_overlay = visualize_mask(
        class_names=example["focus_classes"],
        mask=batch_masks,
        image=batch_predictions["orig_data"],
        img_size=512,
        cat=True,
        axis=1,
    )
    batch_overlay_path = TEN_OUTPUT_DIR / f"{example['dicom_id']}_cxas_overlay.png"
    batch_overlay.save(batch_overlay_path)

    non_empty_classes = int(batch_masks.reshape(batch_masks.shape[0], -1).any(axis=1).sum())
    summary_rows.append(
        {
            "example": index,
            "dicom_id": example["dicom_id"],
            "subject_id": example["subject_id"],
            "study_id": example["study_id"],
            "view": example["view"],
            "focus_area": example["focus_name"],
            "focus_classes": ";".join(example["focus_classes"]),
            "non_empty_classes": non_empty_classes,
            "image_path": str(example["image_path"]),
            "mask_path": str(batch_mask_path),
            "overlay_path": str(batch_overlay_path),
        }
    )
    example_overlays.append((example, batch_overlay.copy()))
    print(
        f"[{index:02d}/10] {example['view']} {example['focus_name']} — "
        f"{non_empty_classes} total non-empty classes"
    )

summary_path = TEN_OUTPUT_DIR / "summary.csv"
with summary_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=summary_rows[0].keys())
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"Saved summary: {summary_path}")

In [ ]:
from PIL import ImageDraw

columns = 2
rows = 5
tile_width = 512
preview_height = 256
caption_height = 32
contact_sheet = Image.new(
    "RGB",
    (columns * tile_width, rows * (preview_height + caption_height)),
    "white",
)
draw = ImageDraw.Draw(contact_sheet)

for position, (example, example_overlay) in enumerate(example_overlays):
    column = position % columns
    row = position // columns
    x = column * tile_width
    y = row * (preview_height + caption_height)
    contact_sheet.paste(example_overlay.resize((tile_width, preview_height)), (x, y))
    caption = f"{position + 1:02d} | {example['view']} | {example['focus_name']}"
    draw.text((x + 8, y + preview_height + 8), caption, fill="black")

contact_sheet_path = TEN_OUTPUT_DIR / "ten_examples_contact_sheet.png"
contact_sheet.save(contact_sheet_path)
print(f"Saved contact sheet: {contact_sheet_path}")
display(contact_sheet)